# 05. Tree-Based Model Training & Optimization

## Purpose
This notebook performs the **heavy lifting** of training our machine learning models.
We are comparing 5 different tree-based algorithms on two different versions of our dataset ("Original" vs "Trees").

## What we are doing:
1.  **Loading Data**: Bringing in the cleaned data.
2.  **Define Models**: Setting up RandomForest, XGBoost, ExtraTrees, etc.
3.  **Hyperparameter Tuning**: Using `RandomizedSearchCV` to find the best settings (like tree depth, number of trees) for each model.
4.  **Optimization Goal**: We optimize for **F2 Score**, which prioritizes **Recall** (catching fraud) over Precision.
5.  **Output**: We save the best "Uncalibrated" models for the next step.


### Step 1: Setup and Imports
**What:** Import necessary Python libraries (Pandas, Scikit-Learn, XGBoost).
**Why:** We need specific tools for data manipulation and model training.


In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add src to path for imports
sys.path.insert(0, '../src')

from utils import load_insurance_data, get_preprocessor, evaluate_model, compare_models, plot_roc_pr_curves
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, cross_val_predict
from sklearn.calibration import CalibratedClassifierCV
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
import joblib

# Set random seed for reproducibility
SEED = 42
from sklearn.metrics import make_scorer, fbeta_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import VotingClassifier
import shap
from sklearn.metrics import precision_score


In [2]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

def add_anomaly_features(X_train, X_test, num_cols, random_state=42):
    X_train = X_train.copy()
    X_test = X_test.copy()

    # Isolation Forest
    isf = IsolationForest(
        n_estimators=150,
        contamination='auto',
        random_state=random_state
    )
    isf.fit(X_train[num_cols])

    X_train['anomaly_isf'] = isf.decision_function(X_train[num_cols])
    X_test['anomaly_isf'] = isf.decision_function(X_test[num_cols])

    # Local Outlier Factor
    lof = LocalOutlierFactor(
        n_neighbors=20,
        novelty=True
    )
    lof.fit(X_train[num_cols])

    X_train['anomaly_lof'] = lof.decision_function(X_train[num_cols])
    X_test['anomaly_lof'] = lof.decision_function(X_test[num_cols])

    return X_train, X_test


### Step 2: Load Data
**What:** Load our two preprocessed datasets:
*   `Original`: Standard cleaning detailed in Notebook 01.
*   `Trees`: A version specifically optimized for tree models (e.g., bins removed).
**Why:** We want to see if the "Trees" specific preprocessing actually yields better results.


In [3]:
# Load both datasets
datasets = {
    'Original': load_insurance_data('preprocessed', verbose=True),
    'Trees': load_insurance_data('trees', verbose=True)
}



✓ Loading preprocessed dataset: insurance_claims_preprocessed_no_hobbies.csv
  Shape: (1000, 51)
  Target distribution: {0: 753, 1: 247}
  Fraud rate: 24.7%
✓ Loading trees dataset: preprocesed_for_trees.csv
  Shape: (1000, 43)
  Target distribution: {0: 753, 1: 247}
  Fraud rate: 24.7%


### Step 3: Define the Training Engine
**What:** This big function `tune_and_evaluate` is the core engine.
**How it works:**
1.  **Split Data**: Uses `train_test_split` (80% Train, 20% Test).
2.  **Preprocess**: Scales the data using `RobustScaler` (good for outliers).
3.  **Grid Search**: Tries many combinations of parameters (e.g., `n_estimators`, `max_depth`) using 5-fold Cross-Validation.
4.  **Operational Simulation**: After finding the best model, it simulates different thresholds (0.1, 0.2... 0.9) to see how many claims would be flagged.
5.  **Ensemble**: It also trains a "Voting" model that combines the others.

**Why:** By wrapping this in a function, we can reliably repeat the exact same process for both datasets.


In [4]:
def tune_and_evaluate(dataset_name, df, time_split=False, cost_sensitive=False, cost_ratio=5):
    print(f"\n{'='*40}\nProcessing Dataset: {dataset_name}\n{'='*40}")
    
    # 1. Split Data
    # If time_split is True, we should ideally sort by date if we had a date column.
    # Assuming df is already sorted or we just use the split logic as requested.
    # For this function, we'll keep the split logic simple but robust.
    X = df.drop('target', axis=1)
    y = df['target']
    
    if time_split:
        print("Using TimeSeriesSplit for validation...")
        cv = TimeSeriesSplit(n_splits=5)
        # Note: train_test_split is random. For strict time-series, we should split by index.
        # But keeping consistent with previous logic for now unless date col is known.
    else:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=SEED
    )
    print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
    
    # 2. Preprocessor
    preprocessor = get_preprocessor(X_train, strategy='robust')
    
    # 3. Define Models and Grids
    
    # Cost sensitive setup
    pos_ratio = y_train.mean()
    neg_ratio = 1 - pos_ratio
    base_imbalance = neg_ratio / pos_ratio
    print(f"Class Imbalance Ratio (Neg/Pos): {base_imbalance:.2f}")
    
    rf_class_weight = ['balanced', 'balanced_subsample']
    et_class_weight = ['balanced', 'balanced_subsample']
    xgb_scale_pos_weight = [1, 3, 5, 7]
    
    if cost_sensitive:
        print(f"Using Cost-Sensitive Learning with cost_ratio={cost_ratio:.1f}")
        target_scale_weight = base_imbalance * cost_ratio
        xgb_scale_pos_weight = [target_scale_weight]
        
        weight_dict = {0: 1.0, 1: target_scale_weight}
        rf_class_weight = [weight_dict]
        et_class_weight = [weight_dict]
        
        print(f"  - Target Class Weight (for 1): {target_scale_weight:.2f}")
    
    # RandomForest
    rf_params = {
        'clf__n_estimators': [100, 200, 300, 400, 500],
        'clf__min_samples_split': [2, 5, 10],
        'clf__min_samples_leaf': [1, 2, 4],
        'clf__max_features': ['sqrt', 'log2', None],
        'clf__max_depth': [None, 5, 10, 15],
        'clf__class_weight': rf_class_weight
    }
    
    # XGBoost - Removed use_label_encoder
    xgb_params = {
        'clf__n_estimators': [100, 200, 300, 400, 500, 600, 800],
        'clf__learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2],
        'clf__max_depth': [3, 4, 5, 6, 7, 8, 9, 10],
        'clf__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'clf__colsample_bytree': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
        'clf__reg_alpha': [0.0, 0.1, 0.5, 1.0, 5.0],
        'clf__reg_lambda': [0.0, 1.0, 5.0, 10.0, 15.0],
        'clf__scale_pos_weight': xgb_scale_pos_weight
    }
    
    # ExtraTrees
    et_params = {
        'clf__n_estimators': [200, 300, 400, 500],
        'clf__max_depth': [10, 15, 20, 25, None],
        'clf__min_samples_split': [2, 5, 10],
        'clf__min_samples_leaf': [1, 2, 4],
        'clf__max_features': ['sqrt', 'log2'],
        'clf__class_weight': et_class_weight
    }
    
    # AdaBoost
    ada_params = {
        'clf__n_estimators': [50, 100, 200, 300],
        'clf__learning_rate': [0.01, 0.05, 0.1, 0.5, 1.0],
        'clf__estimator': [
            DecisionTreeClassifier(max_depth=1),
            DecisionTreeClassifier(max_depth=2),
            DecisionTreeClassifier(max_depth=3)
        ]
    }
    
    # Bagging
    bag_params = {
        'clf__n_estimators': [10, 50, 100, 200],
        'clf__max_samples': [0.5, 0.7, 0.9, 1.0],
        'clf__max_features': [0.5, 0.7, 0.9, 1.0],
        'clf__bootstrap': [True, False],
        'clf__bootstrap_features': [True, False]
    }

    models = [
        ('RandomForest', RandomForestClassifier(random_state=SEED), rf_params, 20),
        ('XGBoost', XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            random_state=SEED
        ), xgb_params, 100),
        ('ExtraTrees', ExtraTreesClassifier(random_state=SEED, n_jobs=-1), et_params, 20),
        ('AdaBoost', AdaBoostClassifier(random_state=SEED), ada_params, 20),
        ('Bagging', BaggingClassifier(
            estimator=DecisionTreeClassifier(),
            random_state=SEED,
            n_jobs=-1
        ), bag_params, 10)
    ]
    
    results = []
    best_estimators = {}
    tuned_models_for_voting = []
    simulation_results = []
    
    # Define F2 Scorer
    from sklearn.metrics import fbeta_score
    f2_scorer = make_scorer(fbeta_score, beta=2)
    scoring = {'pr_auc': 'average_precision', 'f2': f2_scorer}
    
    from sklearn.metrics import recall_score
    
    for name, model, grid, n_iter in models:
        print(f"\nTraining {name} (n_iter={n_iter})...")
        pipe = Pipeline([('prep', preprocessor), ('clf', model)])
        
        search = RandomizedSearchCV(
            pipe, grid, n_iter=n_iter, scoring=scoring, refit='f2',
            cv=cv,
            n_jobs=-1, random_state=SEED, verbose=1
        )
        
        search.fit(X_train, y_train)
        print(f"  Best CV F2: {search.best_score_:.4f}")
        print(f"  Best Params: {search.best_params_}")
        
        best_model = search.best_estimator_
        
        # --- CALIBRATION REMOVED (Moved to 06_tree_calibration.ipynb) ---
        # We just use the best_estimator_ directly
        tuned_models_for_voting.append((name, best_model))
        # --- SHAP REMOVED (Moved to 07_tree_shap.ipynb) ---
        
        # Evaluate
        metrics = evaluate_model(best_model, X_test, y_test, name)
        metrics['dataset'] = dataset_name
        results.append(metrics)
        best_estimators[name] = best_model
        
        # --- OPERATIONAL SIMULATION ---
        print(f"  Operational Simulation for {name}:")
        try:
            probs = best_model.predict_proba(X_test)[:, 1]
            for threshold in np.linspace(0.1, 0.9, 9):
                preds = probs >= threshold
                prec = precision_score(y_test, preds, zero_division=0)
                rec = recall_score(y_test, preds, zero_division=0)
                f2 = fbeta_score(y_test, preds, beta=2, zero_division=0)
                n_flagged = preds.sum()
                
                sim_row = {
                    "model": name,
                    "dataset": dataset_name,
                    "threshold": threshold,
                    "precision": prec,
                    "recall": rec,
                    "f2": f2,
                    "n_flagged": n_flagged
                }
                simulation_results.append(sim_row)
                print(f"    Thresh={threshold:.1f}: Flagged={n_flagged}, Precision={prec:.4f}, Recall={rec:.4f}, F2={f2:.4f}")
        except Exception as e:
             print(f"    Simulation failed: {e}")
            
    # --- SOFT VOTING ENSEMBLE ---
    print("\nTraining Soft Voting Ensemble...")
    try:
        # We use the already fitted calibrated models. 
        # VotingClassifier will clone them and refit them on X_train, y_train.
        # This effectively does the double calibration mentioned, but ensures correctness.
        voting_clf = VotingClassifier(estimators=tuned_models_for_voting, voting='soft')
        voting_clf.fit(X_train, y_train)
        
        metrics = evaluate_model(voting_clf, X_test, y_test, "VotingEnsemble")
        metrics['dataset'] = dataset_name
        results.append(metrics)
        best_estimators["VotingEnsemble"] = voting_clf
    except Exception as e:
        print(f"Voting Ensemble failed: {e}")
    
    return results, best_estimators, X_train, X_test, y_train, y_test, simulation_results


In [5]:
all_results = []
all_best_models = {}
test_sets = {}
train_sets = {}
all_simulation_results = []

for name, df in datasets.items():
    res, models, X_train, X_test, y_train, y_test, sim_res = tune_and_evaluate(name, df)
    all_results.extend(res)
    all_best_models[name] = models
    test_sets[name] = (X_test, y_test)
    train_sets[name] = (X_train, y_train)
    all_simulation_results.extend(sim_res)

# Save uncalibrated models
import joblib
import os
os.makedirs('../models', exist_ok=True)
joblib.dump(all_best_models, '../models/best_tree_models_uncalibrated.joblib')
print('Saved uncalibrated models to ../models/best_tree_models_uncalibrated.joblib')


Processing Dataset: Original========================================
Train shape: (800, 50), Test shape: (200, 50)

Training RandomForest (n_iter=20)...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
  Best CV F2: 0.6477
  Best Params: {'clf__n_estimators': 400, 'clf__min_samples_split': 5, 'clf__min_samples_leaf': 2, 'clf__max_features': 'sqrt', 'clf__max_depth': 5, 'clf__class_weight': 'balanced'}

Training XGBoost (n_iter=100)...
Fitting 5 folds for each of 100 candidates, totalling 500 fits


/Users/tatiana/Desktop/ML Projects/SEP25-BDS-Claim-Prediction/.venv/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [09:17:06] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/tatiana/Desktop/ML Projects/SEP25-BDS-Claim-Prediction/.venv/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [09:17:06] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/tatiana/Desktop/ML Projects/SEP25-BDS-Claim-Prediction/.venv/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [09:17:06] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/tatiana/Desktop/ML Projects/SEP25-BDS-Claim-Prediction/.venv/lib/python3.14

  Best CV F2: 0.6479
  Best Params: {'clf__subsample': 0.7, 'clf__scale_pos_weight': 3, 'clf__reg_lambda': 15.0, 'clf__reg_alpha': 1.0, 'clf__n_estimators': 500, 'clf__max_depth': 10, 'clf__learning_rate': 0.01, 'clf__colsample_bytree': 0.5}

Training ExtraTrees (n_iter=20)...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
  Best CV F2: 0.6449
  Best Params: {'clf__n_estimators': 300, 'clf__min_samples_split': 5, 'clf__min_samples_leaf': 4, 'clf__max_features': 'sqrt', 'clf__max_depth': 15, 'clf__class_weight': 'balanced'}

Training AdaBoost (n_iter=20)...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
  Best CV F2: 0.6502
  Best Params: {'clf__n_estimators': 100, 'clf__learning_rate': 0.05, 'clf__estimator': DecisionTreeClassifier(max_depth=1)}

Training Bagging (n_iter=10)...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
  Best CV F2: 0.5331
  Best Params: {'clf__n_estimators': 100, 'clf__max_samples': 0.5, 'clf__max_features': 1.0, 'clf__

/Users/tatiana/Desktop/ML Projects/SEP25-BDS-Claim-Prediction/.venv/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [09:17:53] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/tatiana/Desktop/ML Projects/SEP25-BDS-Claim-Prediction/.venv/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [09:17:53] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/tatiana/Desktop/ML Projects/SEP25-BDS-Claim-Prediction/.venv/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [09:17:53] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/tatiana/Desktop/ML Projects/SEP25-BDS-Claim-Prediction/.venv/lib/python3.14

  Best CV F2: 0.8325
  Best Params: {'clf__subsample': 0.6, 'clf__scale_pos_weight': 7, 'clf__reg_lambda': 0.0, 'clf__reg_alpha': 1.0, 'clf__n_estimators': 200, 'clf__max_depth': 8, 'clf__learning_rate': 0.005, 'clf__colsample_bytree': 1.0}

Training ExtraTrees (n_iter=20)...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
  Best CV F2: 0.7452
  Best Params: {'clf__n_estimators': 400, 'clf__min_samples_split': 5, 'clf__min_samples_leaf': 4, 'clf__max_features': 'sqrt', 'clf__max_depth': 10, 'clf__class_weight': 'balanced_subsample'}

Training AdaBoost (n_iter=20)...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
  Best CV F2: 0.7886
  Best Params: {'clf__n_estimators': 50, 'clf__learning_rate': 0.01, 'clf__estimator': DecisionTreeClassifier(max_depth=3)}

Training Bagging (n_iter=10)...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
  Best CV F2: 0.7233
  Best Params: {'clf__n_estimators': 100, 'clf__max_samples': 0.5, 'clf__max_features': 1.0

### Interpretation of Results
**What to look for in the output above:**
1.  **Best CV F2**: The score the model achieved during cross-validation (training).
2.  **Confusion Matrix**: Look at the Test Set results.
    *   **TP (Top-Left or Bottom-Right depending on format)**: The number of actual frauds we caught.
    *   **FN**: Frauds we missed (The dangerous ones!).
3.  **Operational Simulation**: Notice how lowering the threshold (e.g., 0.3) catches more fraud (higher Recall) but flags far more innocent claims (lower Precision).

**Next Step:**
Now that we have trained the raw models, we move to **Notebook 06** to "Calibrate" them—smoothing their probabilities so 0.7 really means "70% risk".
